# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and preprocess the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and available via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install mlcroissant if not present
!pip install -q --upgrade mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata details
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and field `@id`s within each record set.


In [ ]:
# Obtain list of record set @id's
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '')}")
    # List field @ids
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      - @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")

## 3. Data Extraction
Extract all available record sets into pandas DataFrames for analysis.

Reference each entity by its `@id` as shown above.


In [ ]:
# Build a list of all record set @ids
record_set_ids = [r['@id'] for r in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[rs_id])} records from record set @id '{rs_id}'. Columns:")
            print(dataframes[rs_id].columns.tolist()[:10])
        else:
            print(f"No records found for record set @id '{rs_id}'.")
    except Exception as e:
        print(f"Could not load records for record set @id '{rs_id}': {e}")

# For demonstration, let's pick the first non-empty record set
main_record_set_id = None
for k,v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if main_record_set_id:
    print(f"\nMain record set selected for EDA: {main_record_set_id}")
else:
    raise ValueError("No non-empty record sets found.")

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We now apply common processing steps, such as filtering records based on numeric field values, normalization, and grouping by categorical fields. All entities are referenced by their `@id`s.

_Please adjust the field `@id` variables below based on the overview above for your analysis._

In [ ]:
# Identify a numeric field and a group field - fill with exact @ids from previous overview as relevant!
numeric_field_id = None
group_field_id = None

# Try to auto-detect numeric and group fields by looking at the dtypes and names
df = dataframes[main_record_set_id]
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) and numeric_field_id is None:
        numeric_field_id = col
    if pd.api.types.is_object_dtype(df[col]) and group_field_id is None:
        group_field_id = col
    if numeric_field_id and group_field_id:
        break

print(f"Using numeric field '@id': {numeric_field_id}")
print(f"Using group field '@id': {group_field_id}")

# Set threshold for filtering
if numeric_field_id:
    # Use 10 as a threshold or the 75th percentile if large values
    try:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head(min(5, len(filtered_df))))
    except Exception as e:
        print(f"Error filtering by numeric field: {e}")

    # Normalize
    try:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    except Exception as e:
        print(f"Error normalizing field: {e}")

    # Group by group_field
    if group_field_id and group_field_id in filtered_df.columns:
        try:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').reset_index()
            print(f"\nMean '{numeric_field_id}' grouped by '{group_field_id}':")
            print(grouped_df.head())
        except Exception as e:
            print(f"Error grouping: {e}")
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its breakdown by the selected categorical field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore the FAIR^2 dataset using the `mlcroissant` library. We loaded metadata, surveyed available record sets and fields using their `@id`s, then performed extraction, processed, and visualized numeric data for initial insights. For more detailed investigations, please consult the full Croissant schema and select fields of scientific relevance.